# Lead Pitcher on Google Colab

Run your **free B2B lead finder + WhatsApp pitching automation** here — no installs on your PC.

**What it does:** you type a city + business type → it finds real businesses (name, address, phone) from OpenStreetMap → you give your criteria as a plain prompt and the AI shortens the list → you review, edit your pitch, approve → scan a WhatsApp QR once → it sends each pitch with safe delays. Nothing is ever sent without your approval.

## Before you run anything

1. **Upload `lead-pitcher.zip` to the ROOT of your Google Drive** (not inside a folder — just open drive.google.com and drag it in).
2. **Get a free Gemini key** (needed for the AI filtering step): go to https://aistudio.google.com/apikey → sign in → **Create API key** → copy it. You'll paste it when asked below (it's typed hidden, never shown or saved anywhere except this Colab machine).

## Honest caveats

- **Colab machines are temporary** — the free tier can disconnect (often within ~12 hours, sooner if idle). If it does, just re-run all cells top to bottom; your app, settings and the "already messaged" list are saved on your Drive and come back. **Nobody ever gets pitched twice**, even across restarts.
- **The web link below is public-but-unguessable** — don't share it with anyone, and shut everything down when you're done.
- **Sending takes real time by design** (45–90 seconds between messages) — keep this Colab tab open while a send run is going.
- **Cost: $0.** Colab free tier, free map data, free Gemini tier, free tunnel.
- **WhatsApp risk reminder:** this uses unofficial WhatsApp automation. Keep volumes low (the app caps at 40/day) — ideally use a separate business number, not your personal one.


## Step 1 — Connect your Google Drive

Run the cell below, click the link, and allow access. This is where the app and your data live.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Step 2 — Install Node.js 20

Colab doesn't come with Node — this installs the official build (takes ~30 seconds).

In [ ]:
%%bash
set -e
if command -v node >/dev/null 2>&1 && node --version | grep -q "v20"; then
  echo "Node 20 already installed: $(node --version)"
else
  echo "Downloading Node v20.20.2..."
  curl -sL -o /tmp/node.tar.xz "https://nodejs.org/dist/v20.20.2/node-v20.20.2-linux-x64.tar.xz"
  tar -xJf /tmp/node.tar.xz -C /usr/local --strip-components=1
  rm /tmp/node.tar.xz
fi
node --version
npm --version

## Step 3 — Install Chrome system libraries

WhatsApp runs inside a real (headless) Chrome — these are its system dependencies.

In [ ]:
%%bash
apt-get update -qq
apt-get install -y -qq libnss3 libatk1.0-0 libatk-bridge2.0-0 libcups2 libdrm2 libxkbcommon0 \
  libxcomposite1 libxdamage1 libxfixes3 libxrandr2 libgbm1 libpango-1.0-0 libcairo2 libasound2
echo "Chrome libs installed.

## Step 4 — Get the app from your Drive

This unzips `lead-pitcher.zip` into `/content/lead-pitcher`.

In [ ]:
%%bash
set -e
ZIP="/content/drive/MyDrive/lead-pitcher.zip"
if [ ! -f "$ZIP" ]; then
  echo "================================================================"
  echo "ERROR: lead-pitcher.zip not found in your Google Drive root."
  echo "Please upload it: open drive.google.com, drag lead-pitcher.zip"
  echo "into the main Drive page (NOT inside a folder), then re-run."
  echo "================================================================"
  exit 1
fi
rm -rf /content/lead-pitcher
mkdir -p /content/lead-pitcher
unzip -q "$ZIP" -d /content/lead-pitcher
# zip contains a top-level lead-pitcher/ folder; flatten it
if [ -f /content/lead-pitcher/lead-pitcher/server.js ]; then
  mv /content/lead-pitcher/lead-pitcher/* /content/lead-pitcher/
  rmdir /content/lead-pitcher/lead-pitcher
fi
ls /content/lead-pitcher
echo "App ready.

## Step 5 — Install app dependencies

Downloads everything the app needs, including Chrome for WhatsApp (a few minutes, run once per machine).

In [ ]:
%%bash
cd /content/lead-pitcher && npm install
echo "Dependencies installed.

## Step 6 — Enter your free Gemini key

Paste the key you created at https://aistudio.google.com/apikey. It is typed hidden, never printed, and only saved on this temporary machine.

In [ ]:
import getpass, os

key = getpass.getpass("Paste your free Gemini API key: ").strip()
if not key or "PASTE" in key:
    raise SystemExit("No valid key entered — please re-run this cell and paste your key.")

with open("/content/lead-pitcher/.env.example") as f:
    template = f.read()
env = template.replace("GEMINI_API_KEY=PASTE_YOUR_FREE_KEY_HERE", "GEMINI_API_KEY=" + key)
with open("/content/lead-pitcher/.env", "w") as f:
    f.write(env)
print("Saved. Key length:", len(key), "chars. (The key itself is never displayed.)")

## Step 7 — Save your data on Drive (survives restarts)

Links the app's database and WhatsApp login to your Drive, so a Colab restart loses nothing. Safe to re-run.

In [ ]:
%%bash
set -e
DRIVE_DATA="/content/drive/MyDrive/lead-pitcher-data"
APP="/content/lead-pitcher"
mkdir -p "$DRIVE_DATA/data" "$DRIVE_DATA/session"

link_dir() {  # $1 = path in app, $2 = path on drive
  if [ -L "$1" ]; then
    echo "Already linked: $1"
  elif [ -d "$1" ]; then
    mv "$1"/* "$2"/ 2>/dev/null || true
    rm -rf "$1"
    ln -s "$2" "$1"
    echo "Linked: $1 -> $2"
  else
    ln -s "$2" "$1"
    echo "Linked: $1 -> $2"
  fi
}

link_dir "$APP/data" "$DRIVE_DATA/data"
link_dir "$APP/.wwebjs_auth" "$DRIVE_DATA/session"
echo "Done. Your leads, send log and WhatsApp login now persist on Drive.

## Step 8 — Start the app + get your web link

This starts the app and gives you a private web address (via Cloudflare Tunnel) to open on your phone or PC.

In [ ]:
%%bash
set -e
# 1. cloudflared binary (free tunnel)
if [ ! -x /usr/local/bin/cloudflared ]; then
  echo "Downloading cloudflared..."
  curl -sL -o /usr/local/bin/cloudflared "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"
  chmod +x /usr/local/bin/cloudflared
fi
cloudflared --version

# 2. start the app
pkill -f "node server.js" 2>/dev/null || true
pkill -f "cloudflared tunnel" 2>/dev/null || true
sleep 2
cd /content/lead-pitcher
nohup node server.js > /content/server.log 2>&1 &
echo "Waiting for the app to start..."
for i in $(seq 1 30); do
  if curl -sf http://localhost:3000/api/health >/dev/null 2>&1; then
    echo "App is up."
    break
  fi
  sleep 2
done
curl -sf http://localhost:3000/api/health || { echo "APP FAILED TO START — see /content/server.log"; tail -20 /content/server.log; exit 1; }

# 3. start the tunnel
nohup cloudflared tunnel --url http://localhost:3000 > /content/cloudflared.log 2>&1 &
sleep 12
URL=$(grep -o 'https://[a-zA-Z0-9.-]*\.trycloudflare\.com' /content/cloudflared.log | head -1)
echo ""
echo "=================================================================="
if [ -n "$URL" ]; then
  echo "  YOUR APP LINK (open in your phone/PC browser):"
  echo ""
  echo "    $URL"
else
  echo "  Tunnel URL not found yet — wait 30s and re-run this cell."
  echo "  (Or check /content/cloudflared.log)"
fi
echo "=================================================================="
echo ""
echo "Keep this Colab tab OPEN while you use the app and while sending.

## Using the app (in the link above)

1. **Find businesses** — enter city (e.g. `Lahore`) + business type (e.g. `hotels`) → takes up to ~1 minute.
2. **Filter with AI** — type your criteria in plain words (e.g. `only hotels in DHA with a website`).
3. **Review** — edit your pitch, uncheck anyone you don't want → **Approve & connect WhatsApp**.
4. **Scan the QR** — click *Show QR code* in the app, scan with WhatsApp on your phone (⋮ menu → Linked devices → Link a device). One time only.
5. **Start sending** — watch the live log. The app waits 45–90s between messages and caps at 40/day. You can stop anytime.
6. **Export CSV** anytime from the campaign page.

## When you're done — shut down

1. In the app, make sure no send run is active (press **Stop** if one is).
2. In Colab: **Runtime → Disconnect and delete runtime**. This kills the app and the tunnel link stops working.
3. Close the tab.

## Troubleshooting

| Problem | Fix |
|---|---|
| Tunnel link shows an error / expired | Colab may have restarted — re-run all cells top to bottom (Runtime → Run all). The link changes every run. |
| QR code won't scan / keeps loading | Make sure your phone has internet; click *Show QR code* in the app again for a fresh code. |
| "Gemini API key is missing" | Re-run Step 6 and paste the key, then re-run Step 8. |
| App page won't load | Check the server log: in a new cell run `!tail -30 /content/server.log`. |
| WhatsApp asks to scan again after restart | Session restore sometimes fails on a fresh machine — just re-scan the QR; your already-messaged list is safe on Drive. |
| "Overpass is busy / timed out" | The free map server is overloaded — wait 1–2 minutes and retry, or search a smaller area like `DHA Lahore`. |
